In [ ]:
%pip install gdown

In [ ]:
%pip install -r requirements.txt

In [ ]:
!apt-get update -qq && apt-get install -y -qq libxrender1 libxext6

In [ ]:
import gdown

file_id = '1ewhkWzZH-uYm1Znn2bo3Z5u1K2xWbhsl'
url = f'https://drive.google.com/uc?id={file_id}'

gdown.download(url, quiet=False)

In [ ]:
import zipfile
from pathlib import Path

zip_path = "processed.zip"
out_dir = "./data/processed"

out_dir = Path(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Extracting {zip_path} -> {out_dir} ...")
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(out_dir)
print("Done.")

## Dataset

In [ ]:
import json
from functools import lru_cache
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset


class QMugsDataset(Dataset):
    def __init__(
        self,
        processed_path,
        split: str = "train",
        split_ratio: tuple = (0.8, 0.1, 0.1),
        seed: int = 42,
        cache_chunks: int = 2,
        max_samples: int = None,
    ):
        processed_path = Path(processed_path)
        self._processed_path = processed_path

        with open(processed_path / "meta.json") as f:
            meta = json.load(f)

        self._chunk_names = meta["chunk_files"]
        chunk_sizes = meta["chunk_sizes"]

        self._chunk_offsets = np.cumsum([0] + chunk_sizes[:-1])
        self._total = sum(chunk_sizes)

        rng = np.random.default_rng(seed)
        perm = rng.permutation(self._total)

        train_end = int(self._total * split_ratio[0])
        val_end = train_end + int(self._total * split_ratio[1])

        splits = {
            "train": perm[:train_end],
            "val":   perm[train_end:val_end],
            "test":  perm[val_end:],
        }
        if split not in splits:
            raise ValueError(f"split must be 'train', 'val', or 'test'; got '{split}'")
        split_indices = splits[split]

        # 청크 순서 셔플 + 청크 내 셔플 → DataLoader shuffle=False로 cache hit 보장
        chunk_of = np.searchsorted(self._chunk_offsets, split_indices, side="right") - 1
        chunk_order = rng.permutation(len(self._chunk_names))
        ordered = []
        for c in chunk_order:
            mask = chunk_of == c
            if mask.any():
                items = split_indices[mask].copy()
                rng.shuffle(items)
                ordered.append(items)
        self.indices = np.concatenate(ordered) if ordered else split_indices
        if max_samples is not None:
            self.indices = self.indices[:max_samples]

        with open(processed_path / "stats.json") as f:
            stats = json.load(f)
        prop_cols = list(stats.keys())
        self.mean = torch.tensor([stats[c]["mean"] for c in prop_cols], dtype=torch.float)
        self.std  = torch.tensor([stats[c]["std"]  for c in prop_cols], dtype=torch.float)

        self._load_chunk = lru_cache(maxsize=cache_chunks)(self._load_chunk_uncached)

        # 원자 수 사전계산 — size bucketing용 (init 1회, 이후 reshuffle에서 재사용)
        self._sizes = self._build_sizes()

    def _load_chunk_uncached(self, chunk_idx: int) -> list:
        return torch.load(
            self._processed_path / self._chunk_names[chunk_idx],
            weights_only=False,
        )

    def _build_sizes(self) -> np.ndarray:
        chunk_of = np.searchsorted(self._chunk_offsets, self.indices, side="right") - 1
        sizes = np.empty(len(self.indices), dtype=np.int32)
        for c in np.unique(chunk_of):
            chunk_data = self._load_chunk(c)
            for pos in np.where(chunk_of == c)[0]:
                local = int(self.indices[pos]) - int(self._chunk_offsets[c])
                sizes[pos] = int((chunk_data[local].z != 1).sum())
        return sizes

    def reshuffle_indices(self):
        # 청크 순서는 랜덤, 청크 내부는 원자 수 오름차순 → size bucketing
        chunk_of = np.searchsorted(self._chunk_offsets, self.indices, side="right") - 1
        chunk_order = np.random.permutation(len(self._chunk_names))
        new_order = []
        for c in chunk_order:
            mask = chunk_of == c
            if mask.any():
                pos = np.where(mask)[0]
                sort = np.argsort(self._sizes[pos], kind="stable")
                new_order.append(pos[sort])
        if new_order:
            perm = np.concatenate(new_order)
            self.indices = self.indices[perm]
            self._sizes = self._sizes[perm]

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int):
        global_idx = int(self.indices[idx])
        chunk_idx = int(np.searchsorted(self._chunk_offsets, global_idx, side="right") - 1)
        local_idx = global_idx - int(self._chunk_offsets[chunk_idx])
        data = self._load_chunk(chunk_idx)[local_idx].clone()
        data.p = (data.p_raw - self.mean) / self.std
        data.a_bin = (data.a[:10] > 0).float()

        # Heavy-atom only: remove H (atomic num 1)
        heavy_mask = data.z != 1
        if not heavy_mask.any():
            heavy_mask[0] = True
        heavy_idx = heavy_mask.nonzero(as_tuple=True)[0]

        old_to_new = torch.full((len(data.z),), -1, dtype=torch.long)
        old_to_new[heavy_idx] = torch.arange(len(heavy_idx), dtype=torch.long)

        ei = data.edge_index
        edge_mask = heavy_mask[ei[0]] & heavy_mask[ei[1]]
        data.z = data.z[heavy_idx]
        data.edge_index = old_to_new[ei[:, edge_mask]]
        data.bond_type = data.bond_type[edge_mask]
        data.num_nodes = len(heavy_idx)

        return data

## Model

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import to_dense_adj, to_dense_batch


MAX_ATOMS = 50
K_X = 10
K_E = 5
T_STEPS = 150

ATOMIC_NUM_TO_CLS = {
    6: 1,  # C
    8: 2,  # O
    7: 3,  # N
    16: 4, # S
    15: 5, # P
    9: 6,  # F
    17: 7, # Cl
    35: 8, # Br
    53: 9  # I
}

NODE_DIM = 256
EDGE_DIM = 64
N_HEADS = 8
N_LAYERS = 6
FF_DIM = 512
T_DIM = 128


def _cosine_ac(T, s = 0.008):
    steps = torch.arange(T+1, dtype=torch.float64)
    f = torch.cos(((steps / T) + s) / (1 + s) * math.pi / 2) ** 2
    return (f / f[0]).float()[1:]


class DiscreteNoiseSchedule(nn.Module):

    def __init__(self, T = T_STEPS):
        super().__init__()
        ac = _cosine_ac(T)
        self.register_buffer("ac", ac) # alpha_bar[1] ... alpha_bar[T]
        self.register_buffer("ac_prev", torch.cat([torch.ones(1), ac[:-1]])) # alpha_bar[0] = 1, alpha_bar[1] ... alpha_bar[T-1]
        self.T = T

    def q_sample(self, x0, t, K):
        shape = x0.shape
        B = t.shape[0]
        ac_t = self.ac[t-1].view(B, *([1] * (len(shape) - 1)))
        keep = torch.bernoulli(ac_t.expand_as(x0.float())).bool()
        return torch.where(keep, x0, torch.randint(0, K, shape, device=x0.device))
    
    def posterior_sample(self, x_t, x0_probs, t, K):
        shape = x_t.shape
        B = t.shape[0]
        view = (B,) + (1,) * (len(shape) - 1)   # for t_mask: (B,1) atoms | (B,1,1) edges
        view_k = (B,) + (1,) * len(shape)        # one extra dim for K after one_hot

        ac_t = self.ac[t-1].view(view_k)
        ac_tm1 = self.ac_prev[t-1].view(view_k)
        alpha_t = (ac_t / ac_tm1.clamp(min=1e-8)).clamp(max=1.0)

        likelihood = alpha_t * F.one_hot(x_t, K).float() + (1 - alpha_t) / K
        prior = ac_tm1 * x0_probs + (1 - ac_tm1) / K

        post = likelihood * prior
        post = post / post.sum(dim=-1, keepdim=True).clamp(min=1e-8)

        stoch = torch.multinomial(post.view(-1, K), 1).view(shape)
        determ = x0_probs.argmax(dim=-1)
        t_mask = (t == 1).view(view).expand(shape)
        return torch.where(t_mask, determ, stoch)
    

class TimeEmbedding(nn.Module):
    
    def __init__(self, dim = T_DIM):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 2), nn.GELU(), nn.Linear(dim * 2, dim)
        )

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device).float() / (half - 1)
        )
        emb = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return self.mlp(emb)
    

class GTLayer(nn.Module):

    def __init__(self):
        super().__init__()
        assert NODE_DIM % N_HEADS == 0

        self.qkv = nn.Linear(NODE_DIM, NODE_DIM * 3, bias=False)
        self.e_bias = nn.Linear(EDGE_DIM, N_HEADS)
        self.attn_out = nn.Linear(NODE_DIM, NODE_DIM)
        self.norm1 = nn.LayerNorm(NODE_DIM)
        self.ff = nn.Sequential(
            nn.Linear(NODE_DIM, FF_DIM), nn.GELU(), nn.Linear(FF_DIM, NODE_DIM)
        )
        self.norm2 = nn.LayerNorm(NODE_DIM)
        self.edge_ff = nn.Sequential(
            nn.Linear(NODE_DIM + EDGE_DIM, FF_DIM),
            nn.GELU(),
            nn.Linear(FF_DIM, EDGE_DIM)
        )
        self.enorm = nn.LayerNorm(EDGE_DIM)

    def forward(self, h, e, pad_mask):
        B, N, _ = h.shape
        nh = N_HEADS
        dh = NODE_DIM // N_HEADS

        Q, K, V = self.qkv(h).chunk(3, dim=-1)
        Q = Q.view(B, N, nh, dh).transpose(1, 2) # (B, nh, N, dh)
        K = K.view(B, N, nh, dh).transpose(1, 2)
        V = V.view(B, N, nh, dh).transpose(1, 2)

        
        scores = torch.einsum("bhid,bhjd->bhij", Q, K) / math.sqrt(dh)  # (B, nh, N, N)
        scores = scores + self.e_bias(e).permute(0, 3, 1, 2)
        scores = scores.masked_fill(pad_mask[:, None, None, :], float('-inf'))

        attn = scores.softmax(dim=-1)
        out  = torch.einsum("bhij,bhjd->bhid", attn, V)
        out = out.transpose(1, 2).reshape(B, N, -1)
        h = self.norm1(h + self.attn_out(out))
        h = self.norm2(h + self.ff(h))

        hi = h.unsqueeze(2).expand(-1, -1, N, -1) # (B, N, N, D)
        hj = h.unsqueeze(1).expand(-1, N, -1, -1) # (B, N, N, D)
        e = self.enorm(e + self.edge_ff(torch.cat([hi + hj, e], dim=-1)))

        h = h.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        e_pad = pad_mask.unsqueeze(1) | pad_mask.unsqueeze(2)
        e = e.masked_fill(e_pad.unsqueeze(-1), 0.0)

        return h, e
    

class ScoreNetwork(nn.Module):

    def __init__(self):
        super().__init__()
        self.x_embed = nn.Embedding(K_X, NODE_DIM)
        self.e_embed = nn.Embedding(K_E, EDGE_DIM)
        self.t_embed = TimeEmbedding(T_DIM)
        self.t_proj  = nn.Linear(T_DIM, NODE_DIM)
        self.layers  = nn.ModuleList([GTLayer() for _ in range(N_LAYERS)])
        self.x_head  = nn.Sequential(nn.LayerNorm(NODE_DIM), nn.Linear(NODE_DIM, K_X))
        self.e_head  = nn.Sequential(nn.LayerNorm(EDGE_DIM), nn.Linear(EDGE_DIM, K_E))

    def forward(self, X_t, E_t, node_mask, t):
        h   = self.x_embed(X_t) + self.t_proj(self.t_embed(t)).unsqueeze(1)
        e   = self.e_embed(E_t)
        pad = ~node_mask

        for layer in self.layers:
            h, e = layer(h, e, pad)

        x_logits = self.x_head(h)
        e_logits = self.e_head(e)
        e_logits = (e_logits + e_logits.transpose(1, 2)) / 2
        return x_logits, e_logits
    

class MoleculeGraphDiffusion(nn.Module):

    def __init__(self, T = T_STEPS):
        super().__init__()
        self.schedule = DiscreteNoiseSchedule(T)
        self.net = ScoreNetwork()
        self.T = T

    def forward(self, batch):
        X0, E0, node_mask = self._to_dense(batch)
        B, N = X0.shape
        device = X0.device

        t = torch.randint(1, self.T + 1, (B,), device=device)
        X_t = self.schedule.q_sample(X0, t, K_X)
        E_t = self._sym(self.schedule.q_sample(E0, t, K_E))

        x_logits, e_logits = self.net(X_t, E_t, node_mask, t)

        nm = node_mask.view(-1)
        loss_x = F.cross_entropy(x_logits.view(-1, K_X)[nm], X0.view(-1)[nm])

        triu = torch.triu(torch.ones(N, N, device=device, dtype=torch.bool), diagonal=1)
        em = (node_mask.unsqueeze(2) & node_mask.unsqueeze(1)) & triu.unsqueeze(0)
        loss_e = F.cross_entropy(
            e_logits.view(-1, K_E)[em.view(-1)],
            E0.view(-1)[em.view(-1)]
        )

        loss = loss_x + loss_e
        return {
            "loss": loss,
            "loss_x": loss_x,
            "loss_e": loss_e
        }
    
    @torch.no_grad()
    def sample(self, n, n_atoms, device):
        node_mask = torch.ones(n, n_atoms, dtype=torch.bool, device=device)
        X_t = torch.randint(0, K_X, (n, n_atoms), device=device)
        E_t = self._sym(torch.randint(0, K_E, (n, n_atoms, n_atoms), device=device))

        for step in range(self.T, 0, -1):
            t = torch.full((n,), step, dtype=torch.long, device=device)
            xl, el = self.net(X_t, E_t, node_mask, t)
            X_t = self.schedule.posterior_sample(X_t, xl.softmax(-1), t, K_X)
            E_t = self._sym(self.schedule.posterior_sample(E_t, el.softmax(-1), t, K_E))

        return X_t, E_t
    
    @staticmethod
    def _sym(E):
        N = E.shape[-1]
        triu = torch.triu(torch.ones(N, N, device=E.device, dtype=torch.bool), diagonal=1)
        E = E * triu
        return E + E.transpose(-1, -2)
    
    @staticmethod
    def _to_dense(batch):
        X0_raw, node_mask = to_dense_batch(batch.z, batch.batch)
        N = X0_raw.size(1)  # batch-local max atom count
        X0 = torch.zeros_like(X0_raw)
        for anum, cls in ATOMIC_NUM_TO_CLS.items():
            X0[X0_raw == anum] = cls

        ei = batch.edge_index
        uv = ei[0] < ei[1]
        bc = batch.bond_type[uv].argmax(dim=-1) + 1
        E0 = to_dense_adj(
            ei[:, uv], batch=batch.batch,
            edge_attr=bc.unsqueeze(-1).float(),
            max_num_nodes=N,
        ).squeeze(-1).long()
        E0 = E0 + E0.transpose(1, 2)

        return X0, E0, node_mask

## Train

In [ ]:
import copy
import io
import sys
import time
from pathlib import Path
from types import SimpleNamespace

import torch
from rdkit import Chem, RDLogger
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch_geometric.loader import DataLoader
from tqdm.notebook import tqdm


RDLogger.DisableLog("rdApp.*")

_BOND_TYPE = {
    1: Chem.rdchem.BondType.SINGLE,
    2: Chem.rdchem.BondType.DOUBLE,
    3: Chem.rdchem.BondType.TRIPLE,
    4: Chem.rdchem.BondType.AROMATIC,
}
_CLS_TO_ANUM = {v: k for k, v in ATOMIC_NUM_TO_CLS.items()}


class _Tee(io.TextIOBase):

    def __init__(self, stream, file_path):
        self._stream = stream
        self._f = open(file_path, "a", encoding="utf-8")

    def write(self, text):
        self._stream.write(text)
        self._f.write(text)
        return len(text)

    def flush(self):
        self._stream.flush()
        self._f.flush()

    def close(self):
        self._f.close()


class EMA:

    def __init__(self, model, decay = 0.9999):
        self.model = copy.deepcopy(model).eval()
        self.decay = decay

    def update(self, model):
        with torch.no_grad():
            for ema_p, p in zip(self.model.parameters(), model.parameters()):
                ema_p.mul_(self.decay).add_(p.data, alpha=1.0 - self.decay)

    def state_dict(self):
        return self.model.state_dict()

    def load_state_dict(self, sd):
        self.model.load_state_dict(sd)


def tensors_to_mol(X, E):
    mol = Chem.RWMol()
    real = (X > 0).nonzero(as_tuple=True)[0].tolist()
    if len(real) < 2:
        return None
    
    idx_map = {}
    for ai in real:
        anum = _CLS_TO_ANUM.get(int(X[ai]))
        if anum is None:
            return None
        idx_map[ai] = mol.AddAtom(Chem.Atom(anum))

    for i, ai in enumerate(real):
        for aj in real[i + 1:]:
            bc = int(E[ai, aj])
            if bc == 0:
                continue
            btype = _BOND_TYPE.get(bc)
            if btype is None:
                continue
            mol.AddBond(idx_map[ai], idx_map[aj], btype)

    try:
        Chem.SanitizeMol(mol)
        return mol.GetMol()
    except Exception:
        return None


def build_size_distribution(dataset, n_sample = 2000):
    sizes = []
    ci = 0
    while len(sizes) < n_sample and ci < len(dataset._chunk_names):
        for data in dataset._load_chunk(ci):
            n = int((data.z != 1).sum())
            if 0 < n <= MAX_ATOMS:
                sizes.append(n)
            if len(sizes) >= n_sample:
                break
        ci += 1
    return sizes


def run_epoch(model, loader, device, train, epoch, optimizer=None, ema=None):
    model.train(train)
    totals = {
        "loss": 0.0,
        "loss_x": 0.0,
        "loss_e": 0.0
    }

    with torch.set_grad_enabled(train):
        for batch in tqdm(loader, desc=f"Epoch {epoch}", leave=False):
            batch = batch.to(device)
            result = model(batch)

            if train:
                optimizer.zero_grad()
                result["loss"].backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1.0)
                optimizer.step()
                if ema is not None:
                    ema.update(model)

            for k in totals:
                totals[k] += result[k].item()

    n = len(loader)
    return {k: v / n for k, v in totals.items()}


@torch.no_grad()
def evaluate_validity(model, size_dist, n_eval, device):
    import random
    from collections import Counter

    model.eval()
    sizes = random.choices(size_dist, k=n_eval)
    size_counts = Counter(sizes)

    valid_mols = []
    for n_atoms, count in size_counts.items():
        if n_atoms > MAX_ATOMS:
            continue
        X, E = model.sample(count, n_atoms, device)
        for i in range(count):
            mol = tensors_to_mol(X[i], E[i])
            if mol is not None:
                valid_mols.append(mol)

    validity = len(valid_mols) / n_eval
    if valid_mols:
        smiles = {Chem.MolToSmiles(m) for m in valid_mols}
        uniqueness = len(smiles) / len(valid_mols)
    else:
        uniqueness = 0.0

    return validity, uniqueness


def main():
    args = SimpleNamespace(
        processed_dir = "./data/processed",
        save_dir = "./checkpoints_diffusion",
        epochs = 200,
        batch_size = 32,
        lr = 1e-4,
        ema_decay = 0.9999,
        save_every = 10,
        eval_every = 10,
        n_eval = 256,
        num_workers = 0,
        max_samples = None,
        log_file = "./train_diffusion.log",
        resume = None,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    save_dir = Path(args.save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    tee = _Tee(sys.stdout, args.log_file)
    sys.stdout = tee
    try:
        print(f"Device: {device}")
        print(f"Log: {args.log_file}")

        print("Loading datasets ...")
        train_set = QMugsDataset(
            args.processed_dir, split="train",
            max_samples=args.max_samples
        )
        val_set = QMugsDataset(
            args.processed_dir, split="val",
            max_samples=args.max_samples // 5 if args.max_samples else None
        )
        train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=False,
                                  num_workers=args.num_workers)
        val_loader   = DataLoader(val_set,   batch_size=args.batch_size, shuffle=False,
                                  num_workers=args.num_workers)
        print(f"  Train: {len(train_set):,}  Val: {len(val_set):,}")

        print("Building size distribution ...")
        size_dist = build_size_distribution(train_set)
        print(f"  Size range: {min(size_dist)} ~ {max(size_dist)} atoms  "
              f"(median {sorted(size_dist)[len(size_dist)//2]})")

        model = MoleculeGraphDiffusion().to(device)
        ema = EMA(model, decay=args.ema_decay)
        ema.model.to(device)

        optimizer = AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
        scheduler = CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=1e-6)

        best_val_loss = float('inf')
        start_epoch = 1

        if args.resume:
            ckpt = torch.load(args.resume, map_location=device)
            model.load_state_dict(ckpt["model"])
            ema.load_state_dict(ckpt["ema"])
            optimizer.load_state_dict(ckpt["optimizer"])
            scheduler.load_state_dict(ckpt["scheduler"])
            start_epoch = ckpt["epoch"] + 1
            best_val_loss = ckpt.get("best_val_loss", float('inf'))
            print(f"  Resumed from epoch {ckpt['epoch']}  (best val loss: {best_val_loss:.4f})")

        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parameters: {n_params:,}")

        print(f"\n{'Epoch':>6}  {'T-loss':>8} {'T-x':>8} {'T-e':>8}  "
              f"{'V-loss':>8} {'V-x':>8} {'V-e':>8}  {'Valid':>7} {'Uniq':>7}  Time")
        print("-" * 105)

        for epoch in range(start_epoch, args.epochs + 1):
            t0 = time.time()
            train_set.reshuffle_indices()

            train = run_epoch(model, train_loader, device, train=True, epoch=epoch, optimizer=optimizer, ema=ema)
            val = run_epoch(ema.model, val_loader, device, train=False, epoch=epoch)

            scheduler.step()

            validity = uniqueness = float("nan")
            if args.eval_every > 0 and epoch % args.eval_every == 0:
                validity, uniqueness = evaluate_validity(ema.model, size_dist, args.n_eval, device)

            elapsed = time.time() - t0
            print(
                f"{epoch:>6}  "
                f"{train['loss']:>8.4f} {train['loss_x']:>8.4f} {train['loss_e']:>8.4f}  "
                f"{val['loss']:>8.4f} {val['loss_x']:>8.4f} {val['loss_e']:>8.4f}  "
                f"{validity:>7.3f} {uniqueness:>7.3f}  {elapsed:>5.1f}s"
            )

            if val["loss"] < best_val_loss:
                best_val_loss = val["loss"]
                torch.save(ema.model.state_dict(), save_dir / "best.pt")

            if args.save_every > 0 and epoch % args.save_every == 0:
                torch.save({
                    "epoch": epoch,
                    "model": model.state_dict(),
                    "ema": ema.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "best_val_loss": best_val_loss
                }, save_dir / f"ckpt_epoch{epoch:04d}.pt")

        print(f"\nDone. Best val loss: {best_val_loss:.4f}")
    finally:
        sys.stdout = tee._stream
        tee.close()


main()

## Test / Inference

In [ ]:
import random
from collections import Counter
from IPython.display import display
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw

RDLogger.DisableLog("rdApp.*")

CKPT_PATH = "./checkpoints_diffusion/best.pt"
N_EVAL = 64
PROCESSED_DIR = "./data/processed"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ema_model = MoleculeGraphDiffusion().to(device)
ckpt = torch.load(CKPT_PATH, map_location=device)
if isinstance(ckpt, dict) and "ema" in ckpt:
    ema_model.load_state_dict(ckpt["ema"])
    print(f"Loaded EMA weights from checkpoint (epoch {ckpt.get('epoch', '?')})")
elif isinstance(ckpt, dict) and "model" in ckpt:
    ema_model.load_state_dict(ckpt["model"])
    print(f"Loaded model weights from checkpoint (epoch {ckpt.get('epoch', '?')})")
else:
    ema_model.load_state_dict(ckpt)
    print("Loaded state dict from best.pt")
ema_model.eval()

_BOND_TYPE = {
    1: Chem.rdchem.BondType.SINGLE,
    2: Chem.rdchem.BondType.DOUBLE,
    3: Chem.rdchem.BondType.TRIPLE,
    4: Chem.rdchem.BondType.AROMATIC,
}
_CLS_TO_ANUM = {v: k for k, v in ATOMIC_NUM_TO_CLS.items()}

def tensors_to_mol(X, E):
    mol = Chem.RWMol()
    real = (X > 0).nonzero(as_tuple=True)[0].tolist()
    if len(real) < 2:
        return None
    idx_map = {}
    for ai in real:
        anum = _CLS_TO_ANUM.get(int(X[ai]))
        if anum is None:
            return None
        idx_map[ai] = mol.AddAtom(Chem.Atom(anum))
    for i, ai in enumerate(real):
        for aj in real[i + 1:]:
            bc = int(E[ai, aj])
            if bc == 0:
                continue
            btype = _BOND_TYPE.get(bc)
            if btype is None:
                continue
            mol.AddBond(idx_map[ai], idx_map[aj], btype)
    try:
        Chem.SanitizeMol(mol)
        return mol.GetMol()
    except Exception:
        return None

def _build_size_dist(processed_dir, n_sample=500, max_samples=3000):
    dataset = QMugsDataset(processed_dir, split="train", max_samples=max_samples)
    sizes = []
    ci = 0
    while len(sizes) < n_sample and ci < len(dataset._chunk_names):
        for data in dataset._load_chunk(ci):
            n = int(data.z.shape[0])
            if n <= MAX_ATOMS:
                sizes.append(n)
            if len(sizes) >= n_sample:
                break
        ci += 1
    return sizes

size_dist = _build_size_dist(PROCESSED_DIR)

sizes = random.choices(size_dist, k=N_EVAL)
size_counts = Counter(sizes)

valid_mols = []
with torch.no_grad():
    for n_atoms, count in size_counts.items():
        if n_atoms > MAX_ATOMS:
            continue
        X, E = ema_model.sample(count, n_atoms, device)
        for i in range(count):
            mol = tensors_to_mol(X[i], E[i])
            if mol is not None:
                valid_mols.append(mol)

validity = len(valid_mols) / N_EVAL
smiles_set = {Chem.MolToSmiles(m) for m in valid_mols}
uniqueness = len(smiles_set) / len(valid_mols) if valid_mols else 0.0

print(f"Validity:   {validity:.3f}  ({len(valid_mols)}/{N_EVAL})")
print(f"Uniqueness: {uniqueness:.3f}")

if valid_mols:
    sampled = random.sample(valid_mols, min(16, len(valid_mols)))
    print(f"\nSMILES (random {min(10, len(valid_mols))}):")
    for mol in random.sample(valid_mols, min(10, len(valid_mols))):
        print(" ", Chem.MolToSmiles(mol))
    img = Draw.MolsToGridImage(sampled, molsPerRow=4, subImgSize=(300, 300))
    display(img)
else:
    print("\n유효한 분자가 생성되지 않았습니다.")
